# 03 — Il confronto (tentativo 10, etichette MBFC)

Sulle 222 testate valutate (le stesse nelle due versioni), fold per gruppo, per testata:

- **classificatore a feature** (TF-IDF + LR, C in validazione interna): voto di maggioranza degli articoli;
- **ModernBERT-base** (un seme): media delle probabilità degli articoli;
- due baseline senza testo (maggioranza, a caso).

Metriche: F1 macro sui 5 livelli MBFC, accuratezza, accuratezza a ±1 livello, binario (almeno *Mostly Factual*).
Intervalli al 95% con un **bootstrap per gruppo** (2000 ricampionamenti dei gruppi); le differenze (ModernBERT −
feature, pulito − grezzo) sugli stessi ricampionamenti: significative se l'intervallo non contiene lo zero.

In [ ]:
import json

import numpy as np
import pandas as pd

from comune import LIVELLI, RESULTS, VERSIONI, VERSIONI_DATASET, metriche_di, per_testata, per_testata_media

B = 2000
OUT = RESULTS / "confronto"; OUT.mkdir(parents=True, exist_ok=True)
fold = pd.read_csv(VERSIONI / "pulito" / "fold.csv").set_index("domain")
val = fold[fold.valutazione]
P = {}
for v in VERSIONI_DATASET:
    f = RESULTS / f"feature_tuned_{v}_fold_gruppo_articoli.csv"
    if f.exists():
        P[("feature", v)] = per_testata(pd.read_csv(f)).y_pred
    f = RESULTS / "transformer" / f"modernbert_{v}_s42" / "articoli.csv"
    if f.exists():
        P[("ModernBERT", v)] = per_testata_media(pd.read_csv(f)).y_pred
assert P, "nessun risultato ancora"
dom = sorted(set.intersection(*[set(p.index) for p in P.values()]))
assert len(dom) == len(val), f"{len(dom)} testate in comune, attese {len(val)}"
y = val.label.reindex(dom).to_numpy(); g = val.gruppo.reindex(dom).to_numpy()
P = {k: p.reindex(dom).to_numpy() for k, p in P.items()}
print(f"{len(dom)} testate valutate, {len(np.unique(g))} gruppi; modelli: {list(P)}")

In [ ]:
gruppi = np.unique(g); pos = {x: np.where(g == x)[0] for x in gruppi}
rng = np.random.default_rng(0)
camp = [np.concatenate([pos[x] for x in rng.choice(gruppi, len(gruppi))]) for _ in range(B)]
METR = ["f1_macro", "accuratezza", "tolerant_1", "bin_accuratezza"]


def boot(p):
    return np.array([[metriche_di(y[i], p[i])[m] for m in METR] for i in camp])


bs = {k: boot(p) for k, p in P.items()}
righe = []
for (mod, v), p in P.items():
    m = metriche_di(y, p); b = bs[(mod, v)]
    r = {"modello": mod, "versione": v}
    for j, k in enumerate(METR):
        r[k] = m[k]; r[f"{k} 2.5%"] = np.percentile(b[:, j], 2.5); r[f"{k} 97.5%"] = np.percentile(b[:, j], 97.5)
    righe.append(r)
for nome, base in [("maggioranza", "baseline_maggioranza"), ("a caso", "baseline_a_caso")]:
    f = RESULTS / f"{base}_fold_gruppo_metriche.json"
    if f.exists():
        m = json.load(open(f))["testata_voto"]
        righe.append({"modello": f"baseline: {nome}", "versione": "-", **{k: m[k] for k in METR}})
tab = pd.DataFrame(righe).set_index(["modello", "versione"]); tab.to_csv(OUT / "intervalli.csv")
tab[METR].round(3)

## Le differenze

In [ ]:
coppie = [(("ModernBERT", v), ("feature", v), f"ModernBERT − feature ({v})") for v in VERSIONI_DATASET] + \
         [((mod, "pulito"), (mod, "grezzo"), f"pulito − grezzo ({mod})") for mod in ["feature", "ModernBERT"]]
righe = []
for a, b, nome in coppie:
    if a in P and b in P:
        d = bs[a] - bs[b]; ma, mb = metriche_di(y, P[a]), metriche_di(y, P[b])
        r = {"confronto": nome}
        for j, k in enumerate(METR):
            r[f"Δ {k}"] = ma[k] - mb[k]; r[f"Δ {k} 2.5%"] = np.percentile(d[:, j], 2.5); r[f"Δ {k} 97.5%"] = np.percentile(d[:, j], 97.5)
        righe.append(r)
diff = pd.DataFrame(righe).set_index("confronto"); diff.to_csv(OUT / "differenze.csv")
diff[[c for c in diff.columns if "f1_macro" in c or "bin_" in c]].round(3)

## Le matrici di confusione (righe: vero; colonne: predetto)

In [ ]:
for k, p in P.items():
    c = pd.crosstab(pd.Series(y).map(dict(enumerate(LIVELLI))), pd.Series(p).map(dict(enumerate(LIVELLI))), rownames=["vero"], colnames=["predetto"])
    c = c.reindex(index=LIVELLI, columns=LIVELLI, fill_value=0)
    c.to_csv(OUT / f"confusione_{k[0]}_{k[1]}.csv")
    print(f"== {k[0]}, {k[1]}"); print(c.to_string()); print()